<a href="https://colab.research.google.com/github/JensenJones/CSS2/blob/DataPreprocessing/CSS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# The previous error indicated that the file was not found at this path.
# Please verify the exact path to your CSV file in Google Drive.
# Ensure the file exists and is accessible at this location.
file_path = '/content/drive/MyDrive/UTS/sports_betting_predictive_analysis.csv'

df = pd.read_csv(file_path)
df.head()

,Match_ID,Date,Sport,Home_Team,Away_Team,Home_Team_Odds,Away_Team_Odds,Draw_Odds,Predicted_Winner,Actual_Winner
0,M00001,2024-01-24,Basketball,Gonzalezmouth Tigers,Hernandezfurt Lions,3.62,3.99,NaN,Hernandezfurt Lions,Hernandezfurt Lions
1,M00002,2025-05-03,Basketball,Rothstad Wolves,North Manuel Eagles,1.84,NaN,NaN,Rothstad Wolves,Rothstad Wolves
2,M00003,2025-04-10,Baseball,Aliciaport Lions,West Gabrielton Lions,3.93,3.29,NaN,Aliciaport Lions,Aliciaport Lions
3,M00004,2024-08-02,Tennis,Lake Samantha Eagles,Simonshire Lions,3.70,3.61,NaN,Simonshire Lions,Lake Samantha Eagles
4,M00005,2024-10-05,Tennis,Brendanport Eagles,Williamsfurt Bears,2.26,2.52,NaN,Williamsfurt Bears,Brendanport Eagles


In [ ]:
# Fill all NaN values in the DataFrame with 0
df.fillna(0, inplace=True)

print("DataFrame after filling NaN values with 0:")
display(df.head())

DataFrame after filling NaN values with 0:


,Match_ID,Date,Sport,Home_Team,Away_Team,Home_Team_Odds,Away_Team_Odds,Draw_Odds,Predicted_Winner,Actual_Winner
0,M00001,2024-01-24,Basketball,Gonzalezmouth Tigers,Hernandezfurt Lions,3.62,3.99,0.0,Hernandezfurt Lions,Hernandezfurt Lions
1,M00002,2025-05-03,Basketball,Rothstad Wolves,North Manuel Eagles,1.84,0.00,0.0,Rothstad Wolves,Rothstad Wolves
2,M00003,2025-04-10,Baseball,Aliciaport Lions,West Gabrielton Lions,3.93,3.29,0.0,Aliciaport Lions,Aliciaport Lions
3,M00004,2024-08-02,Tennis,Lake Samantha Eagles,Simonshire Lions,3.70,3.61,0.0,Simonshire Lions,Lake Samantha Eagles
4,M00005,2024-10-05,Tennis,Brendanport Eagles,Williamsfurt Bears,2.26,2.52,0.0,Williamsfurt Bears,Brendanport Eagles


In [ ]:
import numpy as np

In [ ]:

def calculate_arbitrage(row, total_stake=100):
    """
    Calculate arbitrage for a single row with Home_Team_Odds, Away_Team_Odds, Draw_Odds
    """
    home_odds = row['Home_Team_Odds']
    away_odds = row['Away_Team_Odds']
    draw_odds = row['Draw_Odds']

    # Handle missing or invalid odds
    if pd.isna(home_odds) or pd.isna(away_odds) or pd.isna(draw_odds):
        return None
    if home_odds <= 1 or away_odds <= 1 or draw_odds <= 1:
        return None

    # Convert to implied probabilities
    prob_home = 1 / home_odds
    prob_away = 1 / away_odds
    prob_draw = 1 / draw_odds
    total_prob = prob_home + prob_away + prob_draw

    # Check for arbitrage opportunity
    if total_prob >= 1:
        return {
            'arbitrage_exists': False,
            'total_prob': total_prob,
            'arb_percentage': 0,
            'profit': 0
        }

    # Calculate arbitrage details
    arb_percentage = 1 - total_prob

    # Calculate optimal stakes
    stake_home = total_stake * (prob_home / total_prob)
    stake_away = total_stake * (prob_away / total_prob)
    stake_draw = total_stake * (prob_draw / total_prob)

    # Calculate guaranteed payout and profit
    payout = total_stake / total_prob
    profit = payout - total_stake
    profit_percentage = profit / total_stake * 100

    return {
        'arbitrage_exists': True,
        'total_prob': total_prob,
        'arb_percentage': arb_percentage,
        'profit_percentage': profit_percentage,
        'profit': profit,
        'stake_home': stake_home,
        'stake_away': stake_away,
        'stake_draw': stake_draw,
        'payout': payout,
        'home_odds': home_odds,
        'away_odds': away_odds,
        'draw_odds': draw_odds
    }

In [ ]:
def find_arbitrage_opportunities(df, min_profit_threshold=1.0, total_stake=100):
    """
    Process entire dataset to find arbitrage opportunities

    Parameters:
    df: DataFrame with Home_Team_Odds, Away_Team_Odds, Draw_Odds columns
    min_profit_threshold: Minimum profit percentage to consider (default 1%)
    total_stake: Total investment amount
    """

    # Apply arbitrage calculation to each row
    arbitrage_results = df.apply(lambda row: calculate_arbitrage(row, total_stake), axis=1)

    # Filter out None results and create results DataFrame
    valid_results = [result for result in arbitrage_results if result is not None]

    if not valid_results:
        return pd.DataFrame(), pd.DataFrame()

    results_df = pd.DataFrame(valid_results)
    results_df.index = df.index[:len(results_df)]

    # Combine with original data
    combined_df = pd.concat([df.iloc[:len(results_df)], results_df], axis=1)

    # Filter for profitable arbitrage opportunities
    arbitrage_opportunities = combined_df[
        (combined_df['arbitrage_exists'] == True) &
        (combined_df['profit_percentage'] >= min_profit_threshold)
    ].copy()

    return arbitrage_opportunities, combined_df

In [ ]:
# Example usage with your dataset structure

# --- FIX START: Local redefinition for demonstration purposes ---
# The permanent fix should be applied to the 'calculate_arbitrage' function in cell 6SHGWaStHFou
def calculate_arbitrage(row, total_stake=100):
    """
    Calculate arbitrage for a single row with Home_Team_Odds, Away_Team_Odds, Draw_Odds
    """
    home_odds = row['Home_Team_Odds']
    away_odds = row['Away_Team_Odds']
    draw_odds = row['Draw_Odds']

    # Handle missing or invalid odds
    if pd.isna(home_odds) or pd.isna(away_odds) or pd.isna(draw_odds):
        return None
    if home_odds <= 1 or away_odds <= 1 or draw_odds <= 1:
        return None

    # Convert to implied probabilities
    prob_home = 1 / home_odds
    prob_away = 1 / away_odds
    prob_draw = 1 / draw_odds
    total_prob = prob_home + prob_away + prob_draw

    # Check for arbitrage opportunity
    if total_prob >= 1:
        return {
            'arbitrage_exists': False,
            'total_prob': total_prob,
            'arb_percentage': 0,
            'profit': 0,
            'profit_percentage': 0.0 # ADDED: Ensure 'profit_percentage' is always present
        }

    # Calculate arbitrage details
    arb_percentage = 1 - total_prob

    # Calculate optimal stakes
    stake_home = total_stake * (prob_home / total_prob)
    stake_away = total_stake * (prob_away / total_prob)
    stake_draw = total_stake * (prob_draw / total_prob)

    # Calculate guaranteed payout and profit
    payout = total_stake / total_prob
    profit = payout - total_stake
    profit_percentage = profit / total_stake * 100

    return {
        'arbitrage_exists': True,
        'total_prob': total_prob,
        'arb_percentage': arb_percentage,
        'profit_percentage': profit_percentage,
        'profit': profit,
        'stake_home': stake_home,
        'stake_away': stake_away,
        'stake_draw': stake_draw,
        'payout': payout,
        'home_odds': home_odds,
        'away_odds': away_odds,
        'draw_odds': draw_odds
    }
# --- FIX END ---



=== ARBITRAGE OPPORTUNITIES ===
No arbitrage opportunities found with current thresholds.


In [ ]:
def demo_with_sample_data():
    """Demo function with sample data matching your structure"""

    # Sample data with your column names
    sample_data = {
        'Home_Team_Odds': [2.10, 1.85, 2.50, 3.20, 1.90],
        'Away_Team_Odds': [3.80, 4.20, 2.90, 2.10, 3.50],
        'Draw_Odds': [3.40, 3.60, 3.20, 3.80, 3.40],
        'Home_Team': ['Team A', 'Team B', 'Team C', 'Team D', 'Team E'],
        'Away_Team': ['Team X', 'Team Y', 'Team Z', 'Team W', 'Team V']
    }

    df = pd.DataFrame(sample_data)

    # Find arbitrage opportunities
    arb_opportunities, all_results = find_arbitrage_opportunities(
        df,
        min_profit_threshold=0.5,  # 0.5% minimum profit
        total_stake=1000  # $1000 total stake
    )

    print("=== ARBITRAGE OPPORTUNITIES ===")
    if len(arb_opportunities) > 0:
        for idx, row in arb_opportunities.iterrows():
            print(f"\n🎯 Match {idx}: {row['Home_Team']} vs {row['Away_Team']}")
            print(f"   Profit: ${row['profit']:.2f} ({row['profit_percentage']:.2f}%)वरुन")
            print(f"   Stakes: Home ${row['stake_home']:.2f} | Away ${row['stake_away']:.2f} | Draw ${row['stake_draw']:.2f}")
            print(f"   Odds: {row['Home_Team_Odds']:.2f} | {row['Away_Team_Odds']:.2f} | {row['Draw_Odds']:.2f}")
    else:
        print("No arbitrage opportunities found with current thresholds.")

    return arb_opportunities, all_results

# Run the demo
arb_opps, all_data = demo_with_sample_data()

=== ARBITRAGE OPPORTUNITIES ===
No arbitrage opportunities found with current thresholds.


In [ ]:
def process_your_dataset(file_path, output_file=None):
    """
    Process your actual betting odds dataset
    """
    # Load your dataset
    df = pd.read_csv(file_path)  # or pd.read_excel() if Excel

    print(f"Dataset loaded: {len(df)} rows")
    print("Columns:", df.columns.tolist())

    # Ensure required columns exist
    required_cols = ['Home_Team_Odds', 'Away_Team_Odds', 'Draw_Odds']
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        print(f"❌ Missing columns: {missing_cols}")
        return None, None

    # Find arbitrage opportunities
    print("\n🔍 Scanning for arbitrage opportunities...")

    arb_opportunities, all_results = find_arbitrage_opportunities(
        df,
        min_profit_threshold=0.5,  # Adjust threshold as needed
        total_stake=1000
    )

    print(f"\n📊 RESULTS:")
    print(f"Total matches analyzed: {len(all_results)}")
    print(f"Arbitrage opportunities found: {len(arb_opportunities)}")

    if len(arb_opportunities) > 0:
        avg_profit = arb_opportunities['profit_percentage'].mean()
        max_profit = arb_opportunities['profit_percentage'].max()
        print(f"Average profit: {avg_profit:.2f}%")
        print(f"Maximum profit: {max_profit:.2f}%")

        # Save results if requested
        if output_file:
            arb_opportunities.to_csv(output_file, index=False)
            print(f"Results saved to: {output_file}")

    return arb_opportunities, all_results

In [ ]:
# Call the function to process your dataset
arb_opportunities, all_results = process_your_dataset(file_path)

# Now print the results if they were successfully generated
if arb_opportunities is not None and all_results is not None:
    print("Arbitrage Opportunities:")
    display(arb_opportunities)
    print("\nAll Processed Results:")
    display(all_results)
else:
    print("Could not retrieve arbitrage opportunities or all results. Please check for previous errors.")

Dataset loaded: 1369 rows
Columns: ['Match_ID', 'Date', 'Sport', 'Home_Team', 'Away_Team', 'Home_Team_Odds', 'Away_Team_Odds', 'Draw_Odds', 'Predicted_Winner', 'Actual_Winner']

🔍 Scanning for arbitrage opportunities...

📊 RESULTS:
Total matches analyzed: 443
Arbitrage opportunities found: 154
Average profit: 17.72%
Maximum profit: 53.30%
Arbitrage Opportunities:


,Match_ID,Date,Sport,Home_Team,Away_Team,Home_Team_Odds,Away_Team_Odds,Draw_Odds,Predicted_Winner,Actual_Winner,...,arb_percentage,profit,profit_percentage,stake_home,stake_away,stake_draw,payout,home_odds,away_odds,draw_odds
1,M00002,2025-05-03,Basketball,Rothstad Wolves,North Manuel Eagles,1.84,NaN,NaN,Rothstad Wolves,Rothstad Wolves,...,0.136991,158.736001,15.873600,241.907307,476.846091,281.246602,1158.736001,4.79,2.43,4.12
8,M00009,2024-06-26,Hockey,Lake Karen Bears,Lake Jennifer Lions,1.35,1.89,4.88,Lake Jennifer Lions,Lake Karen Bears,...,0.202537,253.977084,25.397708,278.043699,308.102478,413.853823,1253.977084,4.51,4.07,3.03
10,M00011,2025-01-01,Basketball,New Kathy Tigers,West Melvin Lions,4.89,4.15,NaN,West Melvin Lions,New Kathy Tigers,...,0.158142,187.848427,18.784843,467.656861,245.423229,286.919910,1187.848427,2.54,4.84,4.14
11,M00012,2024-11-03,Hockey,Brownmouth Tigers,Smithmouth Wolves,4.79,2.43,4.12,Draw,Brownmouth Tigers,...,0.329888,492.288718,49.228872,345.437203,308.324115,346.238682,1492.288718,4.32,4.84,4.31
14,M00015,2023-07-31,Baseball,New Emilyport Lions,Port Christopherfurt Tigers,1.94,4.59,NaN,New Emilyport Lions,Port Christopherfurt Tigers,...,0.082865,90.351453,9.035145,364.666038,326.452531,308.881431,1090.351453,2.99,3.34,3.53
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
430,M00431,2025-06-14,Tennis,Teresastad Lions,South Timothytown Lions,NaN,1.73,NaN,South Timothytown Lions,Teresastad Lions,...,0.173813,210.379189,21.037919,355.993879,254.816671,389.189450,1210.379189,3.40,4.75,3.11
437,M00438,2024-10-10,Tennis,Lake Joshuamouth Bears,South Abigail Wolves,4.92,4.73,NaN,South Abigail Wolves,South Abigail Wolves,...,0.024910,25.546609,2.554661,324.540066,231.500363,443.959571,1025.546609,3.16,4.43,2.31
439,M00440,2025-03-25,Football,Davidtown Wolves,New Tonybury Eagles,4.22,2.35,4.45,Draw,Draw,...,0.030167,31.105409,3.110541,266.435506,228.120666,505.443828,1031.105409,3.87,4.52,2.04
440,M00441,2023-09-06,Basketball,Marshallhaven Eagles,Stephensonburgh Lions,4.09,3.90,NaN,Stephensonburgh Lions,Marshallhaven Eagles,...,0.144036,168.273132,16.827313,317.465525,393.357957,289.176518,1168.273132,3.68,2.97,4.04



All Processed Results:


,Match_ID,Date,Sport,Home_Team,Away_Team,Home_Team_Odds,Away_Team_Odds,Draw_Odds,Predicted_Winner,Actual_Winner,...,arb_percentage,profit,profit_percentage,stake_home,stake_away,stake_draw,payout,home_odds,away_odds,draw_odds
0,M00001,2024-01-24,Basketball,Gonzalezmouth Tigers,Hernandezfurt Lions,3.62,3.99,NaN,Hernandezfurt Lions,Hernandezfurt Lions,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M00002,2025-05-03,Basketball,Rothstad Wolves,North Manuel Eagles,1.84,NaN,NaN,Rothstad Wolves,Rothstad Wolves,...,0.136991,158.736001,15.873600,241.907307,476.846091,281.246602,1158.736001,4.79,2.43,4.12
2,M00003,2025-04-10,Baseball,Aliciaport Lions,West Gabrielton Lions,3.93,3.29,NaN,Aliciaport Lions,Aliciaport Lions,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M00004,2024-08-02,Tennis,Lake Samantha Eagles,Simonshire Lions,3.70,3.61,NaN,Simonshire Lions,Lake Samantha Eagles,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M00005,2024-10-05,Tennis,Brendanport Eagles,Williamsfurt Bears,2.26,2.52,NaN,Williamsfurt Bears,Brendanport Eagles,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438,M00439,2024-03-17,Basketball,Williamsfort Eagles,Maxwellborough Bears,2.92,4.01,NaN,Maxwellborough Bears,Williamsfort Eagles,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
439,M00440,2025-03-25,Football,Davidtown Wolves,New Tonybury Eagles,4.22,2.35,4.45,Draw,Draw,...,0.030167,31.105409,3.110541,266.435506,228.120666,505.443828,1031.105409,3.87,4.52,2.04
440,M00441,2023-09-06,Basketball,Marshallhaven Eagles,Stephensonburgh Lions,4.09,3.90,NaN,Stephensonburgh Lions,Marshallhaven Eagles,...,0.144036,168.273132,16.827313,317.465525,393.357957,289.176518,1168.273132,3.68,2.97,4.04
441,M00442,2023-08-08,Tennis,Woodardhaven Wolves,Port Carloshaven Lions,3.42,4.54,NaN,Port Carloshaven Lions,Woodardhaven Wolves,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
